In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as t
from geopy.distance import geodesic

# Инициализация Spark
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("BikeAnalysis") \
    .getOrCreate()

spark

In [ ]:
# Загрузка данных
trips = spark.read.format('csv').option('header', 'true').load("trip.csv")
stations = spark.read.format('csv').option('header', 'true').load("station.csv")

trips = trips.filter(F.col("duration").rlike("^[0-9]+$"))

## 1. Найти велосипед с максимальным временем пробега.

In [ ]:
# Группируем по ID велосипеда и суммируем длительность
bike_max_duration = (
    trips
    .groupBy("bike_id")
    .agg(F.sum(F.col("duration").cast(t.IntegerType())).alias("total_duration"))
    .orderBy(F.col("total_duration").desc())
)
bike_max_duration.show(1)

# Сохраняем ID самого активного велосипеда для третьей задачи
top_bike_id = bike_max_duration.first()["bike_id"]

+-------+--------------+
|bike_id|total_duration|
+-------+--------------+
|    535|      36897410|
+-------+--------------+
only showing top 1 row


## 2. Найти наибольшее геодезическое расстояние между станциями.

In [ ]:
# Подготовка координат
st_data = stations.select(
    F.col("id"),
    F.col("lat").cast(t.DoubleType()),
    F.col("long").cast(t.DoubleType())
)

# Создаем пары всех возможных станций
st_pairs = st_data.alias("a").crossJoin(st_data.alias("b")).filter("a.id < b.id")

# UDF для расчета расстояния через geopy
@F.udf(returnType=t.DoubleType())
def dist_udf(lat_a, lon_a, lat_b, lon_b):
    if all(v is not None for v in [lat_a, lon_a, lat_b, lon_b]):
        return geodesic((lat_a, lon_a), (lat_b, lon_b)).kilometers
    return 0.0

# Вычисляем расстояние и находим максимум
max_geo_dist = (
    st_pairs
    .withColumn("distance_km", dist_udf("a.lat", "a.long", "b.lat", "b.long"))
    .orderBy(F.col("distance_km").desc())
)
max_geo_dist.select("a.id", "b.id", "distance_km").show(1)

+---+---+-----------------+
| id| id|      distance_km|
+---+---+-----------------+
| 16| 60|69.92096757764355|
+---+---+-----------------+
only showing top 1 row


## 3. Найти путь велосипеда с максимальным временем пробега через станции.

In [ ]:
# Хронологический список станций для велосипеда
bike_path = (
    trips
    .filter(F.col("bike_id") == top_bike_id)
    .select("id", "start_station_name", "end_station_name")
    .orderBy(F.col("id").cast(t.IntegerType()))
)
bike_path.show(bike_path.count(), truncate=False)

+------+---------------------------------------------+---------------------------------------------+
|id    |start_station_name                           |end_station_name                             |
+------+---------------------------------------------+---------------------------------------------+
|4966  |Post at Kearney                              |San Francisco Caltrain (Townsend at 4th)     |
|5067  |San Francisco Caltrain (Townsend at 4th)     |San Francisco Caltrain 2 (330 Townsend)      |
|5179  |San Francisco Caltrain 2 (330 Townsend)      |Market at Sansome                            |
|5199  |Market at Sansome                            |2nd at South Park                            |
|7806  |2nd at Townsend                              |Davis at Jackson                             |
|11422 |San Francisco City Hall                      |Civic Center BART (7th at Market)            |
|12245 |Civic Center BART (7th at Market)            |Post at Kearney                      

## 4. Найти количество велосипедов в системе.

In [ ]:
# Подсчет уникальных ID
total_bikes = trips.select("bike_id").distinct().count()
print(f"Количество уникальных велосипедов: {total_bikes}")

Количество уникальных велосипедов: 728


## 5. Найти пользователей потративших на поездки более 3 часов.

In [ ]:
# Группировка по zip_code (используется как идентификатор пользователя)
pro_users = (
    trips
    .groupBy("zip_code")
    .agg(F.sum(F.col("duration").cast(t.IntegerType())).alias("total_time"))
    .filter(F.col("total_time") > 10800) # 3 часа = 10800 секунд
    .orderBy(F.col("total_time").desc())
)
pro_users.show()

+--------+----------+
|zip_code|total_time|
+--------+----------+
|   94107|  92717295|
|     nil|  91449072|
|   94105|  45778916|
|   94133|  39803815|
|    NULL|  37793390|
|   94103|  35622576|
|   94102|  34745604|
|   95531|  34540800|
|   94111|  26297973|
|   95112|  23187505|
|   94109|  21716914|
|   94040|  14841896|
|   94110|  13569520|
|   94117|  12297386|
|   94041|  11694411|
|   94301|  11228137|
|   94158|  11222066|
|   94306|  10429906|
|   94025|   9615755|
|   94108|   9329074|
+--------+----------+
only showing top 20 rows
